# H1. Chọn trước 30 tuyến để khảo sát: 18 trong / 12 ngoài

Từ 60 ứng viên đã có, chọn **30 tuyến đề xuất trước khảo sát**, sau đó đi khảo sát/đo và cập nhật. Không đòi phải có dữ liệu thực địa của cả 60 tuyến mới được chọn. Bộ 30 mang nhãn `proposed_for_survey`, chưa là bộ pháp lý/thực địa đã xác minh.

Dùng Python thường hoặc kernel PyQGIS đã đăng ký; controller gọi worker khi xuất hình học. Bước này không gọi HTTP, không tải lại dữ liệu và không sửa 60 hình học nguồn. Hướng dẫn: `work/docs/lez/SELECTION_30.md`.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

candidates = [Path.cwd(), *Path.cwd().parents]
candidates += [p / "work" for p in candidates]
candidates += [Path.home() / "python" / "work"]
WORK = next((p.resolve() for p in candidates
             if (p / "scripts/lez/selection_30.py").is_file()), None)
if WORK is None:
    raise FileNotFoundError("Không tìm thấy folder work có bộ chọn 30 tuyến.")
if str(WORK / "scripts") not in sys.path:
    sys.path.insert(0, str(WORK / "scripts"))
from lez.selection_30 import preflight_selection, run_selection, selection_status
from lez.common import input_path


## 1. Cấu hình, trọng số và dữ liệu chưa quan sát

Giữ đúng 18 trong/12 ngoài, cụ thể A10/B8/C12, và đủ 3 vùng A/B/C. Dùng 9 biến X; làn/tốc độ có cột quan sát và cột mô hình RC-median riêng, cùng tỷ lệ thiếu. Trọng số C1–C7 là 25/15/15/15/10/10/10.

C6 quan sát vẫn trống. Giả định trung tính 2,5/5 chỉ dùng chấm điểm trước khảo sát; có khoảng điểm rộng 10 điểm khi C6 thực tế nằm trong 0–5. G1 pháp lý và G4–G6 chờ hiện trường. Bộ đề xuất không được coi là bộ đã vượt đầy đủ các điều kiện thực địa.


In [ ]:
OPTIONS = json.loads((WORK / "config/selection_30.json").read_text())
print(json.dumps(OPTIONS, ensure_ascii=False, indent=2))


## 2. Kiểm tra trước khi chọn — không HTTP

Kiểm tra nguồn 60 tuyến, cấu hình, hash và runtime. Vi phạm GIS đã biết được ghi lý do; thiếu quan sát thực địa không chặn lựa chọn trước khảo sát.


In [ ]:
CHECK = preflight_selection(WORK)
print(json.dumps(CHECK, ensure_ascii=False, indent=2))


## 3. Chọn tuần tự và hoán đổi

Mỗi vòng tính lại ΔD, Gap, độ mới hạ tầng, O_max, d_min và vai trò không gian. Tiêu chí C1–C5 dùng min–max trên ứng viên khả dụng, C4 là chi phí. C6 thật còn trống; giả định chấm điểm và khoảng điểm lưu riêng.

Thử 10 seed và hoán đổi có ràng buộc. Giữ quota A10/B8/C12 = 18 trong/12 ngoài. `force=False` dùng cache hợp lệ; kiểm tra độ nhạy mở rộng ở mục 7a–7c.


In [ ]:
RESULT = run_selection(WORK, force=False, progress=print)
print("Trạng thái:", RESULT.get("status"))


## 4. Tiếp tục hoặc đọc kết quả

Sau Interrupt/Ctrl+C, chạy lại cùng cấu hình; giai đoạn đã công bố đầy đủ được dùng lại. `force=True` không cần để tiếp tục thông thường. Báo cáo gần nhất có thể là bản trước khi dừng; đối chiếu thời điểm/hash.


In [ ]:
RESULT = run_selection(WORK, force=False, progress=print)
REPORT = selection_status(WORK)
print(json.dumps(REPORT, ensure_ascii=False, indent=2))


## 5. Xem 30 tuyến được đề xuất và quota thực tế

Đếm dòng/hình học đã xuất, không suy từ quota cấu hình. “Được đề xuất” khác “đã xác minh/khóa để đo”; mọi câu hỏi thực địa được lưu riêng để xử lý sau lựa chọn.


In [ ]:
REPORT = selection_status(WORK)
PATHS = REPORT.get("paths", {})
DIRECTORY = input_path(WORK, PATHS["directory"]) if PATHS.get("directory") else None

def show_table(filename):
    path = DIRECTORY / filename if DIRECTORY is not None else None
    if path is None or not path.is_file():
        print("Chưa có đầu ra:", filename)
        return pd.DataFrame()
    frame = pd.read_csv(path, keep_default_na=False)
    display(frame)
    return frame

SELECTED = show_table("selected_routes.csv")
print("Số tuyến đã xuất trong bảng:", len(SELECTED))
if not SELECTED.empty and "group" in SELECTED:
    display(SELECTED.groupby("group", as_index=False).size())
    print("Đối chiếu yêu cầu 12 trong / 18 ngoài.")
stratum_column = next((c for c in ["stratum_id", "primary_stratum"] if c in SELECTED), None)
if stratum_column:
    display(SELECTED.groupby(stratum_column, as_index=False).size())
    print("Số tầng chính có đại diện:", SELECTED[stratum_column].nunique())
print(json.dumps(PATHS, ensure_ascii=False, indent=2))


## 6. Các biến bắt buộc trong QGIS và dữ liệu quan sát/ước tính

CSV/GPKG chứa đủ 60 tuyến, chiều dài RC, P_ô, làn/tốc độ và tỷ lệ thiếu, mật độ nút/tín hiệu, tỷ lệ một chiều, X và kết quả G2/G3. `lanes` là số làn gắn thẻ OSM; `maxspeed` là giới hạn gắn thẻ, không phải tốc độ đo.

Cột `_scoring_model` chỉ phục vụ chấm điểm. Giữ dữ liệu quan sát và C6 trống khi thiếu. Cell dưới đếm thiếu từ bundle hiện tại; không suy số liệu từ báo cáo cũ.


In [ ]:
VARIABLES = show_table("route_variables.csv")
X_STANDARDIZED = show_table("X_standardized.csv")
print(json.dumps(json.loads((DIRECTORY / "variable_units.json").read_text()), ensure_ascii=False, indent=2))
GATES = json.loads((DIRECTORY / "gate_checks.json").read_text())
print(json.dumps(GATES["summary"], ensure_ascii=False, indent=2))
CLASSIFICATION = show_table("candidate_classification.csv")
RESERVE = show_table("reserve_routes.csv")

MISSING_OBSERVATIONS = pd.DataFrame([
    {"attribute": name,
     "routes_with_no_OSM_value": int((VARIABLES[name] == "").sum()),
     "routes_with_partial_or_no_length": int((pd.to_numeric(VARIABLES[name + "_known_length_fraction"]) < 1 - 1e-9).sum())}
    for name in ["lanes", "maxspeed"]
])
display(MISSING_OBSERVATIONS)


## 7. Lịch sử và kiểm tra độ nhạy trong bundle cơ sở

Điểm lúc chọn thay đổi theo tập đã có. Điểm `final_context_score` là đánh giá thêm lại sau khi bỏ riêng tuyến đó, không phải bảng điểm tĩnh lấy top 30.

Lịch sử lưu giá trị thô, chuẩn hóa, trọng số và ràng buộc. Kịch bản cũ đọc từ `sensitivity.json`; phân tích mở rộng bên dưới chạy lại cùng engine, có z-score và C6 khác nhau giữa tuyến.


In [ ]:
HISTORY = show_table("selection_history.csv")
SWAPS = show_table("swaps.csv")
LEGACY_SENSITIVITY = json.loads((DIRECTORY / "sensitivity.json").read_text())
display(pd.DataFrame(LEGACY_SENSITIVITY)[["scenario", "jaccard", "D_combined", "route_count"]])


## 7a. Kiểm tra trước khi phân tích độ nhạy

Python thường + NumPy; không gọi HTTP hoặc khởi động QGIS. Xác minh hash và cổng GIS đã kiểm tra của bộ 60/bộ 30, rồi tái lập đúng baseline. Cấu hình: `work/config/selection_sensitivity.json`; hướng dẫn: `work/docs/lez/SENSITIVITY.md`.


In [ ]:
from lez.selection_sensitivity import preflight_sensitivity, run_sensitivity, sensitivity_status

SENS_CHECK = preflight_sensitivity(WORK)
print(json.dumps({key: SENS_CHECK[key] for key in ["status", "fingerprint", "network_calls", "qgis_started", "runtime"]}, ensure_ascii=False, indent=2))


## 7b. Chạy hoặc tiếp tục phân tích độ nhạy

Thử min–max/z-score của X, trọng số ±20%, ngưỡng ô 25–40%, bỏ/dịch mô hình làn/tốc độ, C6 1–5 khác nhau giữa tuyến và giả lập C6=0 để tìm tuyến thay thế. Giữ A10/B8/C12, 10 seed và swap. Ba ngưỡng cờ overlap chỉ rà soát, không tính là ba lượt chọn.

Chạy lại cùng cell sau Interrupt/Ctrl+C; cache từng kịch bản được kiểm hash. Bộ 30 hiện hành giữ nguyên. Không ghi C6 giả lập thành quan sát thực địa.


In [ ]:
SENSITIVITY_RESULT = run_sensitivity(WORK, force=False, progress=print)
print(json.dumps({key: SENSITIVITY_RESULT[key] for key in ["status", "total_scenarios", "completed_scenarios", "expected_infeasible_scenarios", "unexpected_infeasible_scenarios", "baseline_reproduced_exactly", "cache_hit"]}, ensure_ascii=False, indent=2))


## 7c. So sánh bộ tuyến và xem tuyến nhạy

`retained` là số tuyến giữ so với baseline, Jaccard là giao/hợp. Xem riêng trước/sau swap và từng nhóm giả định. Tần suất kịch bản không phải xác suất hoặc khoảng tin cậy. Các ca cố ý loại tuyến C6=0 được báo riêng.


In [ ]:
SENSITIVITY_REPORT = sensitivity_status(WORK)
SENS_DIRECTORY = input_path(WORK, SENSITIVITY_REPORT["paths"]["directory"])
SCENARIOS = pd.read_csv(SENS_DIRECTORY / "scenario_summary.csv", keep_default_na=False)
FAMILIES = pd.read_csv(SENS_DIRECTORY / "families.csv", keep_default_na=False)
FREQUENCIES = pd.read_csv(SENS_DIRECTORY / "route_frequency.csv", keep_default_na=False)
REPLACEMENTS = pd.read_csv(SENS_DIRECTORY / "simulated_replacements.csv", keep_default_na=False)
display(FAMILIES)
display(SCENARIOS)
display(FREQUENCIES[FREQUENCIES["scope"] == "all_scoring_stress"].sort_values("selection_fraction"))
display(REPLACEMENTS)
print("Lõi chung của các kịch bản chấm điểm:", SENSITIVITY_REPORT["scoring_common_core"])
print("Báo cáo:", SENSITIVITY_REPORT["paths"]["report"])


## 8. Cơ cấu RC, độ phủ chiều dài và D

Mạng mục tiêu đếm phần đường vật lý một lần; mẫu cộng lượt đi RC. D dùng RC1–RC4, loại connector khỏi mẫu số cơ cấu. P_ô và P_RC trong X chia toàn chiều dài tuyến kể cả connector; `_conditional_RC` dùng chiều dài RC riêng.

Đọc D, độ phủ và số ô trực tiếp từ báo cáo dưới đây. A×RC1 vẫn được giữ trong ma trận/mẫu số dù chưa có tuyến đủ điều kiện; không tuyên bố đại diện đủ 12/12 ô. Xem `work/docs/lez/A_RC1_LIMITATION.md`. D thấp hơn không chứng minh đạt điều kiện thực địa.


In [ ]:
MATRIX = show_table("network_sample_matrix.csv")
print(json.dumps(REPORT, ensure_ascii=False, indent=2))


## 9. Danh sách cần kiểm tra khi đi khảo sát/đo

Bộ 30 đã được đề xuất trước khi có các kết quả này. Kiểm tra quyền đi theo xe/ngày/giờ và chỗ dừng an toàn trước chuyến thực tế; ghi công trình, một chiều/cấm rẽ, làn/tốc độ/tín hiệu, GPS và khả năng lặp. Cập nhật dữ liệu thực địa riêng với nguồn/ngày/chứng cứ; khi thay tuyến, tính lại quota/tầng/D/độ phủ và lưu phiên bản.


In [ ]:
FIELD_CHECKLIST = show_table("field_checklist.csv")


## 10. Bước tiếp theo

Tổ chức khảo sát/đo trên 30 tuyến được đề xuất và giữ các tuyến dự phòng. Cập nhật điểm thực địa C6, quyền đi/neo và thuộc tính quan sát; sửa hoặc thay tuyến khi cần rồi kiểm tra lại tập trước khi khóa kết quả cuối. Số chuyến đo lặp, khung giờ/thiết bị và QA speed–time được tổ chức riêng; 30 tuyến được đề xuất chưa có nghĩa hoàn thành chiến dịch đo.
